<a href="https://colab.research.google.com/github/larasatirida/JogjaPulse/blob/main/secondary_data_boundaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Sekunder & Batas Wilayah

**Proyek:** Group Project Human Mobility — Rekayasa Data
**Dataset utama (Zenodo):** https://zenodo.org/records/23157460 (DIY, 78 kapanewon, periode Des 2021 – Apr 2022)

## Tujuan notebook
1. Menyiapkan **file batas wilayah kapanewon DIY** (GeoJSON) untuk analisis spasial.
2. Menyiapkan **dataset kedua: jumlah penduduk BPS per kapanewon** (2021 & 2022, per jenis kelamin bila tersedia) untuk integrasi anggota tim yang lain.
3. **Memvalidasi** kesesuaian wilayah, nama, dan periode dengan dataset Zenodo, serta menentukan **key join**.

## Output
| File | Fungsi | Dipakai oleh |
|---|---|---|
| `kapanewon_diy_boundary.geojson` | 78 poligon kapanewon DIY (WGS84) | Anggota 6 |
| `kapanewon_lookup.csv` | Kamus `kapanewon_code` ↔ kab/kota ↔ kapanewon | Anggota 3, 4 |
| `gadm_diy.geojson` | Pembanding batas wilayah (GADM 4.1, DIY saja) | dokumentasi |
| `bps_population_kapanewon_diy_2021_2022.csv` | Penduduk per kapanewon, 78 × 2 tahun | Anggota 3, 4, 7 |

**Daftar isi:** (1) Setup → (2) Sumber data → (3) Struktur data batas wilayah → (4) Load GADM → (5) Gabung aaronsanger → (6) Bandingkan nama → (7) CRS & bbox → (8) Keterbatasan batas wilayah → (9) Output batas wilayah → (10) Data penduduk BPS → (11) Validasi dengan Zenodo → (12) Output & kesimpulan

## 1. Setup
Notebook ini bisa dijalankan dari awal sampai akhir di Google Colab. **Ubah path pada sel konfigurasi** sesuai lokasi file di sesi kamu. Library yang dipakai hanya `json`, `re`, `glob`, dan `pandas`.

In [1]:
import json, re, glob, os
import numpy as np
import pandas as pd

AARON_DIR   = "idn-geojson"
GADM_FILE   = "gadm41_IDN_3.json"
BPS_DIR     = "data/raw/bps"
ZENODO_VAL  = "validation_night_presence_kapanewon.csv"
OUT_DIR     = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
if not os.path.isdir(AARON_DIR):
    !git clone --depth 1 https://github.com/aaronsanger/idn-geojson.git
print(sorted(os.listdir(f"{AARON_DIR}/districts"))[:3], "...")
print([os.path.basename(f) for f in sorted(glob.glob(f"{AARON_DIR}/districts/34_*.geojson"))])

['11_01.geojson', '11_02.geojson', '11_03.geojson'] ...
['34_01.geojson', '34_02.geojson', '34_03.geojson', '34_04.geojson', '34_71.geojson']


## 2. Sumber data
| Peran | Sumber | Versi / tanggal | Catatan |
|---|---|---|---|
| **Batas wilayah utama** | [aaronsanger/idn-geojson](https://github.com/aaronsanger/idn-geojson) — folder `districts/34_*.geojson` | export 8 Sep 2026 | Bukan data resmi/terverifikasi. Asal: cahyadsn (rujukan GADM 4.1). Tanggal unduh upstream tidak tercatat. |
| **Batas wilayah pembanding** | [GADM 4.1 level 3](https://gadm.org/download_country.html) (`gadm41_IDN_3.json`) | v4.1 | Lisensi: https://gadm.org/license.html |
| **Penduduk Kota Yogyakarta** | [BPS Kota Yogyakarta](https://jogjakota.bps.go.id/id/statistics-table/1/MzgjMQ==/jumlah-penduduk-menurut-jenis-kelamin-dan-kelurahan-2018-2022.html) | 2018–2022, level kelurahan | Diagregasi ke 14 kecamatan |
| **Penduduk Sleman** | [BPS Kab. Sleman](https://slemankab.bps.go.id/id/statistics-table/2/ODAjMg==/jumlah-penduduk-hasil-sp2020-menurut-kecamatan-dan-jenis-kelamin.html) | 2021, 2022 | L/P/total |
| **Penduduk Kulon Progo** | [BPS Kab. Kulon Progo](https://kulonprogokab.bps.go.id/id/statistics-table/2/NDQ0IzI=/jumlah-penduduk-menurut-kecamatan-dan-jenis-kelamin-tahun-2020.html) | 2021, 2022 | L/P/total |
| **Penduduk Gunungkidul** | [BPS Kab. Gunungkidul](https://gunungkidulkab.bps.go.id/id/statistics-table/2/NTE0IzI=/jumlah-penduduk-pertengahan-tahun-menurut-kapanewon--jiwa-.html) | 2021, 2022 (pertengahan tahun) | **Hanya total**, tidak ada L/P per kapanewon |
| **Penduduk Bantul** | [Satu Data Bantul](https://data.bantulkab.go.id/search/detail?data_id=2.12.0063&ryear=2020+-+2022) | 2020–2022 (2020 = n/a) | L/P/total; angka memakai titik ribuan |

## 3. Struktur data batas wilayah
Dua sumber punya skema atribut berbeda, dan ini penting untuk **key join**:

| Hal | aaronsanger | GADM 4.1 level 3 |
|---|---|---|
| Nama kapanewon | `name` (`KEC. GAMPING`) | `NAME_3` (`Gamping`) |
| Nama kab/kota | `parent_name` (`KAB. SLEMAN`) | `NAME_2` (`Sleman`) |
| Kode wilayah | `code` / `bps_code` (`34.04.01`, **format Kemendagri, sama dengan Zenodo**) | `GID_3` (`IDN.33.1.1_1`, kode GADM sendiri, **tidak bisa dipakai join ke Zenodo**) |
| Provinsi | — | `NAME_1` |

Karena Zenodo memakai kode `34.xx.xx`, **aaronsanger dipilih sebagai batas utama**, GADM hanya pembanding.

## 4. Load GADM dan filter DIY
File GADM berisi **seluruh Indonesia** (6.695 fitur) dalam satu baris panjang (minified), sehingga tidak enak dibuka manual. Kita filter provinsi DIY lewat `NAME_1` yang mengandung "yogya".

In [3]:
if os.path.exists(GADM_FILE):
    gadm = json.load(open(GADM_FILE, encoding="utf-8"))
    print("total fitur seluruh Indonesia:", len(gadm["features"]))
    print("kolom:", list(gadm["features"][0]["properties"].keys()))
    diy_gadm = [f for f in gadm["features"] if "yogya" in f["properties"].get("NAME_1", "").lower()]
    print("fitur DIY di GADM:", len(diy_gadm))
    for f in diy_gadm[:5]:
        p = f["properties"]; print(p["NAME_2"], "|", p["NAME_3"], "|", p["GID_3"])
    json.dump({"type": "FeatureCollection", "features": diy_gadm},
              open(f"{OUT_DIR}/gadm_diy.geojson", "w", encoding="utf-8"))
else:
    diy_gadm = []
    print("GADM_FILE tidak ditemukan -> bagian pembanding GADM dilewati")

total fitur seluruh Indonesia: 6695
kolom: ['GID_3', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'GID_2', 'NAME_2', 'NL_NAME_2', 'NAME_3', 'VARNAME_3', 'NL_NAME_3', 'TYPE_3', 'ENGTYPE_3', 'CC_3', 'HASC_3']
fitur DIY di GADM: 78
Bantul | Bambanglipuro | IDN.33.1.1_1
Bantul | Banguntapan | IDN.33.1.2_1
Bantul | Bantul | IDN.33.1.3_1
Bantul | Dlingo | IDN.33.1.4_1
Bantul | Imogiri | IDN.33.1.5_1


## 5. Gabung 5 file aaronsanger
DIY dibagi per kabupaten/kota: `34_01` Kulon Progo, `34_02` Bantul, `34_03` Gunungkidul, `34_04` Sleman, `34_71` Kota Yogyakarta. Total harus **78** fitur.

In [4]:
aaron = []
for f in sorted(glob.glob(f"{AARON_DIR}/districts/34_*.geojson")):
    feats = json.load(open(f, encoding="utf-8"))["features"]
    print(os.path.basename(f), len(feats))
    aaron += feats
print("fitur aaronsanger:", len(aaron))
assert len(aaron) == 78, "jumlah fitur harus 78"
print("kolom:", sorted(aaron[0]["properties"].keys()))

34_01.geojson 12
34_02.geojson 17
34_03.geojson 18
34_04.geojson 17
34_71.geojson 14
fitur aaronsanger: 78
kolom: ['bps_code', 'code', 'id', 'name', 'parent_id', 'parent_name', 'source']


## 6. Perbandingan nama: aaronsanger vs GADM
Fungsi `norm()` menyamakan ejaan (huruf kecil, buang awalan `KEC.`/`KAB.`, buang spasi dan simbol) lalu kita bandingkan himpunan pasangan **(kab/kota, kapanewon)**.

⚠️ **Catatan penting:** versi awal `norm()` ikut membuang awalan "kota" di *semua* nama, sehingga `Kota Gede` (GADM) menjadi `gede` dan terlihat berbeda dari `KOTAGEDE` (aaronsanger). **Itu bukan perbedaan data, melainkan hanya efek fungsi.** Di bawah, awalan "kota" hanya dibuang pada **nama kab/kota**.

In [5]:
def norm(s, strip_prefix=r"^(kec\.|kab\.)\s*"):
    s = re.sub(strip_prefix, "", str(s).lower().strip())
    return re.sub(r"[^a-z]", "", s)

def norm_kab(s):
    return norm(s, r"^(kab\.|kota)\s*")

A = {(norm_kab(f["properties"]["parent_name"]), norm(f["properties"]["name"])) for f in aaron}
if diy_gadm:
    G = {(norm_kab(f["properties"]["NAME_2"]), norm(f["properties"]["NAME_3"])) for f in diy_gadm}
    print("ada di aaron, tidak di GADM:", sorted(A - G))
    print("ada di GADM, tidak di aaron:", sorted(G - A))
    print("cocok:", len(A & G), "dari", len(A))

ada di aaron, tidak di GADM: []
ada di GADM, tidak di aaron: []
cocok: 78 dari 78


## 7. CRS dan bounding box
**CRS (Coordinate Reference System)** adalah aturan yang menjelaskan arti angka koordinat.
- **EPSG:4326 (WGS84)**: derajat lon/lat (contoh `110.36, -7.79`). Standar GeoJSON dan GPS. **Batas wilayah kami memakai ini.**
- **UTM (mis. EPSG:32749)**: meter (contoh `393000, 9127500`). Dipakai kolom `x_min_utm`/`y_min_utm` pada file grid 500 m di Zenodo.

Dua layer dengan CRS berbeda **tidak akan tumpang tindih** bila tidak disamakan (mis. `gdf.to_crs(4326)` di geopandas). File GeoJSON ini tidak mencantumkan `crs`, sehingga standarnya WGS84; kita buktikan lewat rentang koordinat (*bounding box*) yang harus berada di kisaran DIY.

In [6]:
def bbox(feats):
    xs, ys = [], []
    def walk(c):
        if isinstance(c[0], (int, float)): xs.append(c[0]); ys.append(c[1])
        else: [walk(i) for i in c]
    for f in feats: walk(f["geometry"]["coordinates"])
    return min(xs), max(xs), min(ys), max(ys)

print("aaron bbox (lon_min, lon_max, lat_min, lat_max):", bbox(aaron))
if diy_gadm: print("GADM  bbox:", bbox(diy_gadm))

lo, hi, la, lb = bbox(aaron)
assert 109.9 < lo < hi < 111 and -8.5 < la < lb < -7.4

aaron bbox (lon_min, lon_max, lat_min, lat_max): (110.00352917503147, 110.83881134626665, -8.205622635775967, -7.541161559258512)
GADM  bbox: (110.0139, 110.8346, -8.204, -7.5418)


## 8. Keterbatasan data batas wilayah
- **Tahun/versi tidak tercatat.** README aaronsanger hanya mencantumkan tanggal export (8 Sep 2026); asal data upstream (cahyadsn, rujukan GADM 4.1) tanpa tanggal unduh. Batas kapanewon DIY relatif stabil, tetapi ini tetap keterbatasan untuk periode 2021–2022.
- **Bukan sumber resmi** (bukan BIG/BPS) dan tidak diverifikasi independen.
- **Nama kapanewon tidak unik** antar kabupaten (mis. `JETIS` di Bantul dan Kota Yogyakarta). Join harus memakai **kode wilayah** atau pasangan **kab/kota + kapanewon**.
- **Lisensi:**
> - Proyek Utama: https://github.com/aaronsanger/idn-geojson.git
> - Pembanding Proyek Utama: https://gadm.org/download_country.html
> - Lisensi GADM: https://gadm.org/license.html

**Kesimpulan:** aaronsanger dipilih sebagai batas utama (78 fitur, nama cocok dengan Zenodo, ada kode `34.xx.xx`); GADM menjadi pembanding independen yang hasilnya konsisten.

## 9. Output batas wilayah
Menambah kolom bersih (`kapanewon_clean`, `kab_kota_clean`, `join_key`) lalu menyimpan **GeoJSON final** dan **tabel kamus**.

In [7]:
rows = []
for x in aaron:
    p = x["properties"]
    p["kapanewon_clean"] = re.sub(r"^KEC\.?\s*", "", p["name"]).strip()
    p["kab_kota_clean"]  = re.sub(r"^(KAB\.|KOTA)\s*", "", p["parent_name"]).strip()
    p["join_key"]        = p["code"]
    rows.append({"kapanewon_code": p["code"], "kab_kota": p["kab_kota_clean"], "kapanewon": p["kapanewon_clean"]})

lookup = pd.DataFrame(rows).sort_values("kapanewon_code").reset_index(drop=True)
assert lookup.kapanewon_code.is_unique and len(lookup) == 78
dup = lookup[lookup.duplicated("kapanewon", keep=False)]
print("nama kapanewon kembar antar kab/kota:\n", dup.to_string() if len(dup) else "tidak ada")

json.dump({"type": "FeatureCollection", "features": aaron},
          open(f"{OUT_DIR}/kapanewon_diy_boundary.geojson", "w", encoding="utf-8"))
lookup.to_csv(f"{OUT_DIR}/kapanewon_lookup.csv", index=False)
lookup.head()

nama kapanewon kembar antar kab/kota:
    kapanewon_code    kab_kota kapanewon
20       34.02.09      BANTUL     JETIS
65       34.71.02  YOGYAKARTA     JETIS


,kapanewon_code,kab_kota,kapanewon
0,34.01.01,KULON PROGO,TEMON
1,34.01.02,KULON PROGO,WATES
2,34.01.03,KULON PROGO,PANJATAN
3,34.01.04,KULON PROGO,GALUR
4,34.01.05,KULON PROGO,LENDAH


## 10. Data penduduk BPS (dataset kedua)
Lima sumber, **lima format berbeda**, sehingga tiap sumber punya parser sendiri, lalu disatukan menjadi satu tabel **long**: 1 baris = 1 kapanewon × 1 tahun.

| Sumber | Kendala format | Penanganan |
|---|---|---|
| Kulon Progo, Sleman | Header 4 baris, baris total kabupaten di bawah | `skiprows` dan buang baris total |
| Gunungkidul | Hanya total (tanpa L/P) | L/P diisi kosong (`NaN`), **tidak dikarang** |
| Bantul | Format panjang bertingkat berdasarkan kode; angka bertitik ribuan (`956.513`), `n/a` | Parse berdasar jumlah titik di kolom `Kode`, hapus titik ribuan |
| Kota Yogyakarta | Level **kelurahan**; kolom kecamatan hanya terisi di baris pertama; ejaan `KEC.Matrijeron`/`KEC Jetis` tidak konsisten | `ffill`, buang awalan, **jumlahkan ke kecamatan**, koreksi `Matrijeron → Mantrijeron` |

In [8]:
def simple_csv(path, kab, year, cols):
    d = pd.read_csv(path, header=None, skiprows=4 if len(cols) > 2 else 3, encoding="utf-8-sig")
    d = d.iloc[:, :len(cols)]; d.columns = cols
    d = d[~d.kecamatan.str.lower().str.contains("kabupaten|^gunungkidul$", regex=True)]
    return d.assign(kab_kota=kab, tahun=year)

BPS_DIR = "."

parts = []
for y in (2021, 2022):
    parts.append(simple_csv(f"{BPS_DIR}/Jumlah Penduduk Menurut Kecamatan dan Jenis Kelamin, {y}.csv",
                            "KULON PROGO", y, ["kecamatan", "laki_laki", "perempuan", "total"]))
    parts.append(simple_csv(f"{BPS_DIR}/Jumlah Penduduk Menurut Kecamatan dan Jenis Kelamin di Kabupaten Sleman, {y}.csv",
                            "SLEMAN", y, ["kecamatan", "laki_laki", "perempuan", "total"]))
    gk = simple_csv(f"{BPS_DIR}/Jumlah Penduduk Pertengahan Tahun Menurut Kapanewon, {y}.csv",
                    "GUNUNGKIDUL", y, ["kecamatan", "total"])
    gk["laki_laki"] = np.nan; gk["perempuan"] = np.nan
    parts.append(gk)
print([len(p) for p in parts])

[12, 17, 18, 12, 17, 18]


In [9]:
bantul_file = glob.glob(f"{BPS_DIR}/*Bantul*.csv")[0]
b = pd.read_csv(bantul_file, dtype=str)
b["lvl"] = b["Kode"].str.count(r"\.")
to_num = lambda v: pd.to_numeric(str(v).replace(".", ""), errors="coerce")

recs, cur = [], None
for _, r in b.iterrows():
    if r.lvl == 3: cur = r["Nama Data"]
    if r.lvl in (3, 4):
        for y in (2021, 2022):
            recs.append((cur, "Total" if r.lvl == 3 else r["Nama Data"], y, to_num(r[str(y)])))
bt = (pd.DataFrame(recs, columns=["kecamatan", "jk", "tahun", "v"])
        .pivot_table(index=["kecamatan", "tahun"], columns="jk", values="v").reset_index()
        .rename(columns={"Laki-laki": "laki_laki", "Perempuan": "perempuan", "Total": "total"}))
bt["kab_kota"] = "BANTUL"
parts.append(bt[["kecamatan", "laki_laki", "perempuan", "total", "kab_kota", "tahun"]])
print(bt.groupby("tahun").total.agg(["count", "sum"]))

       count       sum
tahun                 
2021      17  956513.0
2022      17  964245.0


In [10]:
ky = pd.read_csv(f"{BPS_DIR}/Jumlah Penduduk Menurut Jenis Kelamin dan Kelurahan, 2018-2022.csv",
                 header=None, skiprows=4, encoding="utf-8-sig").iloc[:, :9]
ky[1] = ky[1].ffill()
ky = ky[ky[2].notna()]
print("jumlah kelurahan:", len(ky))

for y, (l, p, t) in {2022: (3, 4, 5), 2021: (6, 7, 8)}.items():
    d = ky[[1, l, p, t]].copy(); d.columns = ["kecamatan", "laki_laki", "perempuan", "total"]
    for c in ["laki_laki", "perempuan", "total"]: d[c] = pd.to_numeric(d[c], errors="coerce")
    d["kecamatan"] = d.kecamatan.str.replace(r"^KEC\.?\s*", "", regex=True).str.strip()
    d = d.groupby("kecamatan", as_index=False).sum(min_count=1)
    parts.append(d.assign(kab_kota="YOGYAKARTA", tahun=y))

jumlah kelurahan: 45


### Menyatukan dan memetakan ke kode kapanewon
Nama BPS dicocokkan ke `kapanewon_lookup` memakai kunci **(kab/kota, nama ternormalisasi)**, bukan nama saja karena ada nama kembar. Hasil `merge(..., indicator=True)` harus **semua `both`**.

In [11]:
df = pd.concat(parts, ignore_index=True)
for c in ["laki_laki", "perempuan", "total"]: df[c] = pd.to_numeric(df[c], errors="coerce")

key = lambda s: re.sub(r"[^a-z]", "", str(s).lower())
df["k"] = df.kecamatan.map(key).replace({"matrijeron": "mantrijeron"})
lk = lookup.assign(k=lookup.kapanewon.map(key))

m = df.merge(lk, on=["kab_kota", "k"], how="outer", indicator=True, suffixes=("_bps", ""))
print(m["_merge"].value_counts().to_dict())
assert (m["_merge"] == "both").all(), m[m["_merge"] != "both"]

{'both': 156, 'left_only': 0, 'right_only': 0}


### Cek konsistensi internal
Pada sumber yang punya L dan P, **L + P seharusnya = total**. Bila selisih > 1, baris diberi **`flag_total_mismatch`**. Untuk baris yang punya L dan P, kolom `total` = L + P (nilai asli dari BPS disimpan di `total_published`); untuk Gunungkidul `total` = nilai BPS.

In [12]:
m["total_published"] = m["total"]
lp = m["laki_laki"] + m["perempuan"]
m["total"] = np.where(lp.notna(), lp, m["total_published"])
m["flag_total_mismatch"] = lp.notna() & ((lp - m["total_published"]).abs() > 1)

print("baris tidak konsisten:", int(m.flag_total_mismatch.sum()))
m.loc[m.flag_total_mismatch, ["kab_kota", "kapanewon", "tahun", "laki_laki", "perempuan", "total_published", "total"]]

baris tidak konsisten: 7


,kab_kota,kapanewon,tahun,laki_laki,perempuan,total_published,total
133,YOGYAKARTA,GONDOKUSUMAN,2021,20295.0,22344.0,43179.0,42639.0
137,YOGYAKARTA,JETIS,2021,13245.0,14761.0,27286.0,28006.0
141,YOGYAKARTA,KRATON,2021,10598.0,11163.0,21755.0,21761.0
144,YOGYAKARTA,MERGANGSAN,2022,15258.0,16326.0,31404.0,31584.0
145,YOGYAKARTA,MERGANGSAN,2021,15343.0,16444.0,31875.0,31787.0
152,YOGYAKARTA,UMBULHARJO,2022,34696.0,36241.0,71937.0,70937.0
154,YOGYAKARTA,WIROBRAJAN,2022,13609.0,14224.0,27863.0,27833.0


In [13]:
sumber = {
 "BANTUL": "Satu Data Bantul (kode 2.12.0063)",
 "GUNUNGKIDUL": "BPS Kab. Gunungkidul - Penduduk Pertengahan Tahun menurut Kapanewon",
 "KULON PROGO": "BPS Kab. Kulon Progo - Penduduk menurut Kecamatan dan Jenis Kelamin",
 "SLEMAN": "BPS Kab. Sleman - Penduduk menurut Kecamatan dan Jenis Kelamin",
 "YOGYAKARTA": "BPS Kota Yogyakarta - Penduduk menurut Jenis Kelamin dan Kelurahan (diagregasi)"}
m["sumber"] = m.kab_kota.map(sumber)

pop = (m[["kapanewon_code", "kab_kota", "kapanewon", "tahun", "laki_laki", "perempuan",
          "total", "total_published", "flag_total_mismatch", "sumber"]]
       .sort_values(["kapanewon_code", "tahun"]).reset_index(drop=True))
pop["tahun"] = pop["tahun"].astype(int)
for c in ["laki_laki", "perempuan", "total", "total_published"]: pop[c] = pop[c].astype("Int64")

assert len(pop) == 156 and pop.kapanewon_code.nunique() == 78
print(pop.groupby("tahun").total.sum())
pop.groupby(["kab_kota", "tahun"]).agg(n_kapanewon=("kapanewon", "nunique"), total=("total", "sum"))

tahun
2021    3695058
2022    3713247
Name: total, dtype: Int64


n_kapanewon    total
kab_kota    tahun                      
BANTUL      2021            17   956513
            2022            17   964245
GUNUNGKIDUL 2021            18   748269
            2022            18   749707
KULON PROGO 2021            12   438195
            2022            12   440674
SLEMAN      2021            17  1136474
            2022            17  1145854
YOGYAKARTA  2021            14   415607
            2022            14   412767

## 11. Validasi dengan Zenodo (`pop_actual`)
Zenodo sudah punya kolom `pop_actual` per kapanewon. Kita bandingkan dengan total BPS **2021** (awal periode data mobilitas) untuk mengetahui seberapa dekat dan dari mana perbedaannya. Rasio `total_BPS / pop_actual` ≈ 1 berarti angkanya senada.

In [14]:
if os.path.exists(ZENODO_VAL):
    z = pd.read_csv(ZENODO_VAL).drop_duplicates(["kab_kota", "kapanewon"])[["kab_kota", "kapanewon", "pop_actual"]]
    c = pop[pop.tahun == 2021].merge(z, on=["kab_kota", "kapanewon"], how="left")
    c["rasio"] = c.total / c.pop_actual
    print("kapanewon tanpa pasangan di Zenodo:", int(c.pop_actual.isna().sum()))
    print(c.groupby("kab_kota").agg(bps_2021=("total", "sum"), pop_actual=("pop_actual", "sum"),
                                    rasio_min=("rasio", "min"), rasio_median=("rasio", "median"), rasio_max=("rasio", "max")).round(3))
else:
    print("ZENODO_VAL tidak ditemukan -> bagian validasi dilewati")

kapanewon tanpa pasangan di Zenodo: 0
             bps_2021  pop_actual  rasio_min  rasio_median  rasio_max
kab_kota                                                             
BANTUL         956513    991890.0      0.908           1.0      1.024
GUNUNGKIDUL    748269    748270.0      0.996           1.0      1.001
KULON PROGO    438195    438190.0        1.0           1.0        1.0
SLEMAN        1136474   1134150.0      1.002         1.002      1.002
YOGYAKARTA     415607    375280.0      1.037         1.144      1.208


**Cara membaca:** Sleman, Kulon Progo, dan Gunungkidul hampir identik dengan `pop_actual` (rasio ≈ 1,00), sehingga datanya senada. **Kota Yogyakarta** (rasio median ~1,15) dan sebagian **Bantul** (Banguntapan, Kasihan, Sewon: ~0,91) berbeda cukup jauh, artinya `pop_actual` kemungkinan memakai sumber atau tahun yang berbeda di dua wilayah itu.

## 12. Simpan output dan kesimpulan

In [15]:
pop.to_csv(f"{OUT_DIR}/bps_population_kapanewon_diy_2021_2022.csv", index=False)
print(sorted(os.listdir(OUT_DIR)))

['bps_population_kapanewon_diy_2021_2022.csv', 'gadm_diy.geojson', 'kapanewon_diy_boundary.geojson', 'kapanewon_lookup.csv']


### Kesimpulan
- **Batas wilayah:** 78 poligon (aaronsanger), nama dan jumlah cocok dengan GADM dan Zenodo; CRS WGS84; **key join = `kapanewon_code`** (atau pasangan kab/kota + kapanewon).
- **Data penduduk:** 78 kapanewon × 2 tahun (2021, 2022). L/P tersedia untuk 4 dari 5 wilayah (Gunungkidul hanya total).
- **Catatan kualitas:** 7 baris Kota Yogyakarta memiliki L+P ≠ total di sumber (ditandai `flag_total_mismatch`); `pop_actual` Zenodo berbeda dari BPS 2021 untuk Kota Yogyakarta dan sebagian Bantul.
- **Saran periode:** gunakan **2021** (awal periode Des 2021 – Apr 2022); 2022 tersedia sebagai pembanding.
- **Keterbatasan:** tahun batas wilayah tidak tercatat, sumber bukan resmi BIG/BPS, tahun BPS (2021/2022) tidak persis sama dengan rentang Zenodo.